# 08. 액면가로 판정하니 감자가 통째로 빠졌다

> 2026-09-04 · 이동원 · 결론 문서:
> [데이터 품질 규약 v3.3](../../docs/데이터파트/version3.3/데이터_품질_규약.md)

어제 종목기본정보 9,220,879행 수집을 마쳤습니다. 오늘은 그것을 **전량으로** 검사하려고
했는데, 검사를 만드는 과정에서 **검사 자체가 두 번 틀렸습니다.**

이 노트북은 그 두 번을 순서대로 재현합니다.

| 무엇 | 적어 둔 값 | 전량 실측 |
|---|---:|---:|
| 이름으로 우선주를 추측할 때 어긋나는 종목 | 7종 | **10종 · 10,190행** |
| 수정주가가 못 이은 자리 | 0 (v9 완료 보고) | **17자리** |

두 실수의 모양이 같습니다. **한 축을 전량으로 봐도 다른 축이 표본이면 그건 여전히
표본입니다.**

In [1]:
import sqlite3
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent.parent))
import pandas as pd

from common.paths import krx_db_path

db = sqlite3.connect(krx_db_path())
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

전체 = pd.read_sql(
    "SELECT COUNT(*) 행, COUNT(DISTINCT bas_dd) 거래일, COUNT(DISTINCT code) 종목, "
    "MIN(bas_dd) 처음, MAX(bas_dd) 마지막 FROM stock_base_info", db)
전체

,행,거래일,종목,처음,마지막
0,9220879,4101,3677,20100104,20260831


---

## 1. 첫 번째 실수 — 세 날짜만 봤다

우선주를 이름으로 추측하면(`'우'` 로 끝나거나 `우B`·`우C`·`우(전환)` 를 품으면) 어긋나는
종목이 있습니다. 그 근거를 만들 때 **20150102 · 20200102 · 20260901 세 날짜**만 봤고,
7종이 나왔습니다.

그런데 그건 *"적어도 7종"* 이지 *"정확히 7종"* 이 아닙니다.

먼저 **하루만 전수로** 봅니다. `stock_base_info` 의 마지막 날인 **20260831** 유가 전 종목입니다.
(20260901 은 시세의 마지막 거래일이라 다음 거래일을 몰라 `known_at` 을 못 냅니다 —
아직 안 받았습니다.)

In [2]:
COMMON = "보통주"


def 이름추측_우선주(name: str) -> bool:
    n = (name or "").strip()
    return bool(n) and (n.endswith("우") or "우B" in n or "우(전환)" in n
                        or n.endswith("우C"))


하루 = pd.read_sql(
    "SELECT code, isu_abbrv, kind_stkcert_tp_nm FROM stock_base_info "
    "WHERE bas_dd='20260831' AND market='KOSPI'", db)
하루["추측"] = 하루["isu_abbrv"].map(이름추측_우선주)
하루["정본"] = 하루["kind_stkcert_tp_nm"] != COMMON
어긋 = 하루[하루["추측"] != 하루["정본"]]

print(f"20260831 유가 {len(하루):,}종 전수 — 어긋남 {len(어긋)}건")
어긋

20260831 유가 943종 전수 — 어긋남 0건


,code,isu_abbrv,kind_stkcert_tp_nm,추측,정본


### 🔴 하루를 **전수로** 세도 0건입니다

종목 축을 전량으로 봤습니다. 943종을 하나도 안 빠뜨렸습니다. 그런데 0건입니다.

**빠진 축은 날짜입니다.** 어긋남은 대부분 *개명 구간* 에만 생깁니다 — 미래에셋대우는
2016~2021 에만 그 이름이었고, 지금은 그 이름이 아닙니다. 오늘 하루만 보면 없는 일입니다.

이제 **전 구간**을 봅니다.

In [3]:
어긋남 = pd.read_sql(f"""
    SELECT code, isu_abbrv, kind_stkcert_tp_nm,
           MIN(bas_dd) 처음, MAX(bas_dd) 마지막, COUNT(*) 일수
      FROM stock_base_info
     WHERE (kind_stkcert_tp_nm = '{COMMON}') = (
               isu_abbrv LIKE '%우' OR isu_abbrv LIKE '%우B%'
               OR isu_abbrv LIKE '%우(전환)%' OR isu_abbrv LIKE '%우C')
     GROUP BY code, isu_abbrv, kind_stkcert_tp_nm
     ORDER BY 일수 DESC""", db)

# SQL 의 LIKE 로 좁히고 판정은 파이썬 함수로 다시 한다 — 두 규칙이 갈라지면
# 검사 자체가 거짓말을 하므로 최종 판정은 한 곳에서만 한다.
어긋남 = 어긋남[
    어긋남.apply(lambda r: 이름추측_우선주(r["isu_abbrv"])
                 != (r["kind_stkcert_tp_nm"] != COMMON), axis=1)]
어긋남["방향"] = 어긋남["kind_stkcert_tp_nm"].map(
    lambda k: "추측=우선주·정본=보통주" if k == COMMON else "추측=보통주·정본=우선주")

print(f"어긋난 (종목,이름) 조합 {len(어긋남)} · 걸린 종목-일수 {어긋남['일수'].sum():,}행")
어긋남[["code", "isu_abbrv", "방향", "처음", "마지막", "일수"]].reset_index(drop=True)

어긋난 (종목,이름) 조합 10 · 걸린 종목-일수 10,190행


,code,isu_abbrv,방향,처음,마지막,일수
0,115960,연우,추측=우선주·정본=보통주,20151102,20240305,2051
1,088910,동우,추측=우선주·정본=보통주,20100104,20170417,1806
2,025620,신우,추측=우선주·정본=보통주,20100104,20160414,1556
3,294090,이오플로우,추측=우선주·정본=보통주,20200914,20260831,1460
4,006800,미래에셋대우,추측=우선주·정본=보통주,20160527,20210405,1193
5,047050,포스코대우,추측=우선주·정본=보통주,20160329,20190327,733
6,064960,S&T대우,추측=우선주·정본=보통주,20100104,20120402,562
7,458650,성우,추측=우선주·정본=보통주,20241031,20260831,446
8,159910,에코글로우,추측=우선주·정본=보통주,20250520,20260831,314
9,000327,디피아이홀딩스2B,추측=보통주·정본=우선주,20100104,20100412,69


**7종이 아니라 10종입니다.** 새로 나온 셋을 보면 왜 놓쳤는지 알 수 있습니다.

| 종목 | 왜 안 보였나 |
|---|---|
| 047050 **포스코대우** | `~대우` 라 '우'로 끝난다. 2016~2019 에만 그 이름이었다 |
| 064960 **S&T대우** | 같은 이유. 2010~2012 |
| 000327 **디피아이홀딩스2B** | **반대 방향** — 이름은 보통주처럼 보이는데 정본이 우선주다. 69일뿐 |

앞의 둘은 `대우` 라는 회사명이 우연히 '우'로 끝난 경우입니다. 마지막 하나는 이름 규칙이
아예 못 잡는 모양입니다.

> **그래서 유니버스는 이름이 아니라 정본(`kind_stkcert_tp_nm`)으로 가릅니다.**
> `supply.top_by_market_cap` 이 그 정문입니다.

---

## 2. 두 번째 실수 — 액면가로 자본변동을 판정했다

어제 수정주가(`adj_*` 4칸)를 9,223,644행 전부에 채웠습니다. 그때 액면분할을 대조했고
통과했습니다.

오늘 검사를 만들면서 **상장주식수가 크게 변한 자리**를 보기로 했습니다. 첫 판정 규칙은
*"액면가가 같이 바뀌었나"* 였습니다. 액면분할이면 그렇게 잡히니까요.

In [4]:
급변 = pd.read_sql("""
    WITH t AS (
      SELECT code, isu_abbrv, bas_dd,
             CAST(list_shrs AS INTEGER) s,
             LAG(CAST(list_shrs AS INTEGER)) OVER w p,
             parval v, LAG(parval) OVER w pv,
             LAG(bas_dd) OVER w pd
        FROM stock_base_info
      WINDOW w AS (PARTITION BY code ORDER BY bas_dd))
    SELECT code, isu_abbrv, pd 전일, bas_dd 당일, p 전주식수, s 후주식수,
           pv 전액면가, v 후액면가
      FROM t WHERE p > 0 AND s > 0 AND (s*1.0/p >= 10 OR p*1.0/s >= 10)""", db)

급변["액면가도바뀜"] = 급변["전액면가"] != 급변["후액면가"]
print(f"주식수가 10배 이상 변한 자리 {len(급변)}")
print(급변["액면가도바뀜"].value_counts().rename({True: "액면가도 바뀜", False: "액면가는 그대로"}))

주식수가 10배 이상 변한 자리 420
액면가도바뀜
액면가는 그대로    219
액면가도 바뀜     201
Name: count, dtype: int64


### 🔴 219건이 "설명 안 됨" 으로 쏟아졌습니다

절반이 넘습니다. 이쯤 되면 **설명이 틀린 것**을 의심해야 합니다 — 자료가 절반이나
이상할 리는 없습니다.

무엇인지 보겠습니다.

In [5]:
설명안됨 = 급변[~급변["액면가도바뀜"]].copy()
설명안됨["배수"] = (설명안됨["후주식수"] / 설명안됨["전주식수"]).round(3)
설명안됨.sort_values("당일", ascending=False).head(8)[
    ["code", "isu_abbrv", "전일", "당일", "전주식수", "후주식수", "배수", "후액면가"]]

,code,isu_abbrv,전일,당일,전주식수,후주식수,배수,후액면가
310,083660,CSA 코스믹,20260803,20260804,106638983,7109265,0.067,200
352,145210,다이나믹디자인,20260723,20260724,42236668,4223666,0.100,500
364,200230,텔콘RF제약,20260629,20260630,71523533,7152353,0.100,1000
220,042040,케이피엠테크,20260625,20260626,24709362,2470936,0.100,1000
371,208640,썸에이지,20260624,20260625,139240254,13924025,0.100,100
419,900300,오가닉티코스메틱,20260623,20260624,389004876,7780097,0.020,무액면
332,109960,앱토크롬,20260529,20260601,220789269,11039463,0.050,500
101,006740,블루산업개발,20260528,20260529,56975588,4747965,0.083,500


**전부 0.1배 안팎입니다 — 주식수가 1/10 로 줄었습니다.** 그런데 액면가는 그대로입니다.

이것이 **감자(무상감자)** 입니다. 액면분할·병합과 다릅니다.

| | 주식수 | 액면가 |
|---|---|---|
| 액면분할 (1:2) | ×2 | ÷2 |
| 액면병합 (10:1) | ÷10 | **×10** |
| **무상감자 (10:1)** | ÷10 | **그대로** |

실제 기사도 그렇게 씁니다 — *"보통주 10주를 **동일 액면가**의 1주로 병합하는 무상감자"*
(케이피엠테크, 2026-08). 위 표에 042040 케이피엠테크가 그대로 있습니다.

**액면가는 분할·병합만 가려냅니다.** 감자에는 눈이 없습니다.

---

## 3. 판정을 바꾼다 — 가격이 이론대로 뛰었나

주식수가 x 배가 되면 주가는 **1/x 배**가 되어야 합니다. 10:1 감자면 주가가 10배입니다.

수정주가는 그 점프를 **이어 붙여 없애야** 합니다. 그러니 이렇게 물으면 됩니다.

> `close` 가 이론 점프를 따라갔는데 `adj_close` 도 **같이** 따라갔는가?

따라갔다면 조정이 안 된 것입니다. 임계값도 10배에서 **2배**로 내립니다 —
5:1·3:1 감자를 통째로 놓치고 있었습니다.

In [6]:
후보 = pd.read_sql("""
    WITH t AS (
      SELECT code, isu_abbrv, market, kind_stkcert_tp_nm 종류, bas_dd,
             CAST(list_shrs AS INTEGER) s,
             LAG(CAST(list_shrs AS INTEGER)) OVER w p,
             LAG(bas_dd) OVER w pd
        FROM stock_base_info
      WINDOW w AS (PARTITION BY code ORDER BY bas_dd))
    SELECT code, isu_abbrv, market, 종류, pd 전일, bas_dd 당일, p 전주식수, s 후주식수
      FROM t WHERE p > 0 AND s > 0 AND (s*1.0/p >= 2 OR p*1.0/s >= 2)""", db)
print(f"주식수가 2배 이상 변한 자리 {len(후보):,} (10배 기준은 {len(급변)}이었다)")

# 시세를 한 번에 끌어와 메모리에서 맞춘다 — 건마다 질의하면 2천 번을 왕복한다
필요 = (set(zip(후보["code"], 후보["전일"], strict=True))
        | set(zip(후보["code"], 후보["당일"], strict=True)))
시세 = {}
for c, d, close, adj in db.execute(
        "SELECT code, bas_dd, close, adj_close FROM daily_price"):
    if (c, d) in 필요:
        시세[(c, d)] = (close, adj)
print(f"맞댈 시세 {len(시세):,}쌍")

주식수가 2배 이상 변한 자리 2,048 (10배 기준은 420이었다)


맞댈 시세 4,095쌍


In [7]:
줄 = []
for r in 후보.itertuples():
    a, b = 시세.get((r.code, r.전일)), 시세.get((r.code, r.당일))
    if not a or not b or not all([a[0], a[1], b[0], b[1]]):
        continue
    이론 = r.전주식수 / r.후주식수          # 주식수가 1/10 이면 주가는 10배
    close배, adj배 = b[0] / a[0], b[1] / a[1]
    if abs(close배 / 이론 - 1) > 0.30:
        continue                            # close 가 안 튀었다 — 증자·전환 등 정상
    줄.append({**r._asdict(), "이론": round(이론, 3),
               "close배": round(close배, 2), "adj배": round(adj배, 2),
               "못이음": abs(adj배 / 이론 - 1) < 0.30})

자본변동 = pd.DataFrame(줄).drop(columns=["Index"])
print(f"가격이 이론대로 뛴 자리 {len(자본변동):,}")
print(자본변동["못이음"].value_counts().rename({False: "수정주가가 이었다", True: "🔴 못 이었다"}))

가격이 이론대로 뛴 자리 1,064
못이음
수정주가가 이었다    1047
🔴 못 이었다        17
Name: count, dtype: int64


In [8]:
미조정 = 자본변동[자본변동["못이음"]].sort_values("당일")
미조정[["code", "isu_abbrv", "market", "종류", "전일", "당일",
        "전주식수", "후주식수", "adj배"]].reset_index(drop=True)

,code,isu_abbrv,market,종류,전일,당일,전주식수,후주식수,adj배
0,017170,훈영,KOSPI,보통주,20110401,20110404,53406538,2670326,20.00
1,004555,대우송도개발1우,KOSPI,구형우선주,20120511,20120514,1000000,143386,5.00
2,056810,위다스,KOSDAQ,보통주,20130326,20130327,21393160,7131053,3.01
3,176440,에이치엔티,KOSDAQ,보통주,20191120,20191121,17147869,85739345,0.19
4,050090,비케이홀딩스,KOSDAQ,보통주,20230504,20230508,79138942,19784735,4.96
5,139050,시티랩스,KOSDAQ,보통주,20230627,20230628,106494006,7862472,15.42
6,110790,크리스에프앤씨,KOSDAQ,보통주,20231019,20231020,11715480,23430960,0.50
7,078860,아이오케이,KOSDAQ,보통주,20231226,20231227,96126472,4806323,19.71
8,001465,BYC우,KOSPI,구형우선주,20240416,20240417,215385,2153850,0.09
9,009415,태영건설우,KOSPI,구형우선주,20240711,20240712,1302142,649974,2.00


### 🔴 17자리에서 수정주가가 원가격과 똑같이 뛰었습니다

훈영(20110404)은 **20배**입니다. 그 날 수익률이 **+1,900%** 로 읽히고 라벨이 상승으로
뒤집힙니다.

출처를 보면 더 분명합니다.

In [9]:
출처 = pd.read_sql(
    "SELECT adj_source, COUNT(*) 행 FROM daily_price GROUP BY adj_source", db)
print(출처.to_string(index=False))

쿼리 = " OR ".join(f"(code='{r.code}' AND bas_dd='{r.당일}')" for r in 미조정.itertuples())
pd.read_sql(f"SELECT code, bas_dd, close, adj_close, adj_source "
            f"FROM daily_price WHERE {쿼리} ORDER BY bas_dd", db)

adj_source       행
     chain 1701213
       fdr 7522431


,code,bas_dd,close,adj_close,adj_source
0,017170,20110404,2100,2100.0,fdr
1,004555,20120514,14750,14750.0,fdr
2,056810,20130327,1040,1040.0,fdr
3,176440,20191121,2500,2500.0,fdr
4,050090,20230508,1964,1964.0,fdr
5,139050,20230628,6570,6570.0,fdr
6,110790,20231020,9450,9450.0,fdr
7,078860,20231227,3840,8455.0,fdr
8,001465,20240417,17100,17100.0,fdr
9,009415,20240712,7130,7130.0,fdr


**전부 `fdr` 입니다.** 우리가 계산한 것(`chain`)이 아니라 FinanceDataReader 가 준 값이
그대로 안 이어져 있습니다.

FDR 의 공식 입장은 이렇습니다.

> *"FinanceDataReader 에서 가져오는 모든 가격 데이터는 수정 주가(Adjusted Price)입니다.
> 따라서 종가(Close) 역시 수정종가(Adjusted Close) 입니다."* — Issue #21

**라이브러리 문서를 근거로 삼으면 안 됩니다. 자기 자료로 재야 합니다.**

---

## 4. 지금 얼마나 아픈가 — 유니버스에 드는지 본다

결함의 크기는 행 수가 아니라 **그 행이 쓰이는지**로 재야 합니다.
우리 유니버스는 `supply.top_by_market_cap` 이고 기본값이 **시장별 시총 상위 50** 입니다.

In [10]:
순위들 = []
for r in 미조정.itertuples():
    n = db.execute("""
        SELECT COUNT(*)+1 FROM daily_price d
          JOIN stock_base_info b ON b.code=d.code AND b.bas_dd=d.bas_dd
         WHERE d.bas_dd=? AND b.market=? AND b.kind_stkcert_tp_nm='보통주'
           AND d.close*CAST(b.list_shrs AS INTEGER) >
               (SELECT d2.close*CAST(b2.list_shrs AS INTEGER) FROM daily_price d2
                  JOIN stock_base_info b2 ON b2.code=d2.code AND b2.bas_dd=d2.bas_dd
                 WHERE d2.code=? AND d2.bas_dd=?)""",
        (r.당일, r.market, r.code, r.당일)).fetchone()[0]
    순위들.append({"code": r.code, "이름": r.isu_abbrv, "시장": r.market,
                   "날짜": r.당일, "종류": r.종류, "시장내_시총순위": n})

순위 = pd.DataFrame(순위들).sort_values("시장내_시총순위")
보통주 = 순위[순위["종류"] == "보통주"]
print(f"유니버스(시장별 상위 50)에 든 것 : {(보통주['시장내_시총순위'] <= 50).sum()}건")
print(f"보통주 중 가장 높은 순위        : {보통주['시장내_시총순위'].min()}위")
순위.reset_index(drop=True)

유니버스(시장별 상위 50)에 든 것 : 0건
보통주 중 가장 높은 순위        : 239위


,code,이름,시장,날짜,종류,시장내_시총순위
0,176440,에이치엔티,KOSDAQ,20191121,보통주,239
1,110790,크리스에프앤씨,KOSDAQ,20231020,보통주,344
2,007460,에이프로젠,KOSPI,20260508,보통주,650
3,003060,에이프로젠바이오로직스,KOSPI,20260508,보통주,768
4,017170,훈영,KOSPI,20110404,보통주,776
5,004555,대우송도개발1우,KOSPI,20120514,구형우선주,789
6,001465,BYC우,KOSPI,20240417,구형우선주,802
7,012170,아센디오,KOSPI,20260828,보통주,828
8,009310,참엔지니어링,KOSPI,20260511,보통주,829
9,012170,아센디오,KOSPI,20250306,보통주,838


### 지금은 한 건도 안 걸립니다. 그렇다고 없는 문제는 아닙니다

| 물음 | 답 |
|---|---|
| 유니버스(상위 50)에 드나 | **0건** |
| 보통주 중 가장 높은 순위 | **239위** (에이치엔티 · KOSDAQ) |
| 언제부터 걸리나 | 유니버스를 **상위 239위**까지 넓히면 |
| 전 종목으로 피처를 만들면 | **지금도 걸립니다** |

우선주 4건은 유니버스가 보통주만 담으므로 애초에 대상이 아닙니다.

**그리고 이 문제는 앞으로 커집니다.**

| 무엇 | 수치 |
|---|---:|
| 2026년 액면병합 | **276건** |
| 상장폐지 개혁(2026-02) 이후 7월 중순까지 주식병합 | **256건** (작년 같은 기간 10건) |

주가 1,000원 미만이 30거래일 이어지면 관리종목이 되는 제도가 2026-07 시행됐고,
동전주가 병합·감자로 대응하고 있습니다. **최근 구간일수록 위험합니다.**

---

## 5. 그래서 검사가 매번 이걸 본다

`scripts/verify_base_info.py` 9절이 위 판정을 그대로 담고 있습니다.
그리고 **반출 경로가 이 검사를 스스로 돌립니다.**

`scripts/upload_to_hf.py` 는 private 확인 뒤에 검증기 셋을 돌리고, 하나라도 붉으면
종료코드 1 로 멈춥니다. 오늘 실제로 막혔습니다.

```
── 반출 전 검사 ──
  ✅ 종목 식별 (법인번호·ISIN·상장일)            ── 판정 ── ✅ 이상 없음
  🔴 종목기본정보 (주권종류·자본변동·자리표시자)   · 자본변동을 수정주가가 못 이은 자리 17
  ✅ 업종 스냅샷 (값 대조·지수 조인)              ── 판정 ── ✅ 이상 없음

🔴 중단 — 검사 1건이 붉다. 이대로 올리면 팀원이 그대로 쓴다.
```

"고쳤다"와 "나갔다"는 다릅니다. 사람이 기억해서 돌리는 검사는 바쁠 때 건너뜁니다.

---

## 6. 오늘 배운 것

**축이 둘이면 표본도 둘로 넓혀야 합니다.**

우선주 검사는 종목 축을 943종 전수로 봤는데 0건이었습니다. 날짜 축이 하루였기 때문입니다.
자본변동 검사는 2,048자리를 전수로 봤는데 219건이 "설명 안 됨"으로 쏟아졌습니다.
판정 축이 액면가 하나였기 때문입니다.

그래서 검사를 쓸 때마다 **"이 검사가 안 보는 축은 무엇인가"** 를 주석으로 적기로 했습니다.

**그리고 극단을 미리 자르지 않습니다.** 감자 미조정 17자리는 극단이라서 눈에 띄었습니다.
윈저화로 미리 잘라 두었다면 영영 못 찾았을 것입니다.

> 자세한 규약: [데이터 품질 규약 v3.3](../../docs/데이터파트/version3.3/데이터_품질_규약.md)